In [2]:
sam1=SAM()
sam1.load_data('../../SAM_AC_ncbi_soupx_cleaned_03122025.h5ad')

In [1]:
!pip install anndata==0.8.0

In [2]:
!pip install loompy

In [1]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import loompy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sam1.adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,n_counts,n_genes,key,leiden_clusters,region_label,neurotransmitter,nCount_SCT,...,eq_subclass_nounlabeled_NN,eq_subclass_nounlabeled_nmm,ss_subclass,ss_subclass_nounlabeled,ss_class,ss_subclass_nounlabeled_astro,ss_subclass_v2,ss_subclass_v2_nounlabeled,ss_subclass_v3_nounlabeled,ss_subclass_nounlabeled_nmm
AAACCCAGTTTGGAAA,AC,2108.0,1113,2108.0,1113,Run12_sample1,1,Outside hypothalamus,unclear,2807.0,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,31 OPC-Oligo,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
AAACGAAAGGACTGGT,AC,2050.0,1078,2050.0,1078,Run12_sample1,26,Unknown,Gaba,2832.0,...,ac_17,cj_ac_xt_2,Unlabeled,ac_20,unknown,ac_20,Unlabeled,ac_15,ac_15,ac_xt_dr_1
AAACGAACAAGAGATT,AC,4400.0,2075,4400.0,2075,Run12_sample1,22,Unknown,Gaba,3810.0,...,ac_10,ac_10,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,20 MB GABA,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba
AAACGAACATGACGTT,AC,2572.0,1336,2572.0,1336,Run12_sample1,0,Outside hypothalamus,unclear,2905.0,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,31 OPC-Oligo,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
AAACGCTTCCTAGCGG,AC,7505.0,2390,7505.0,2390,Run12_sample1,61,Outside hypothalamus,unclear,4082.0,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,31 OPC-Oligo,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCATATGGC,AC,2901.0,1549,2901.0,1549,Run17_sample4,41,Unknown,Gaba,3488.0,...,ac_13,cj_ac_9,Unlabeled,ac_11,unknown,ac_11,Unlabeled,ac_11,ac_11,ac_11
TTTGTTGTCCAAACCA,AC,8108.0,3072,8108.0,3072,Run17_sample4,49,Unknown,Glut,4892.0,...,ac_29,ac_29,Unlabeled,ac_12,unknown,ac_12,Unlabeled,ac_20,ac_17_20,ac_xt_dr_4
TTTGTTGTCGCAGTCG,AC,2290.0,1251,2290.0,1251,Run17_sample4,28,POA,Gaba,3408.0,...,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,12 HY GABA,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba
TTTGTTGTCTACTCAT,AC,2500.0,1452,2500.0,1452,Run17_sample4,48,Unknown,Glut,3395.0,...,ac_10,ac_10,175 SC Bnc2 Glut,175 SC Bnc2 Glut,19 MB Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut


In [3]:
for i in range(0,30):
    dat = sc.read_loom('../../subset_Allen_institute_Full_subclass_test_250_'+str(i)+'.loom')
    dat.obs_names = dat.obs['obs_names']
    dat.var_names = list(dat.var['x'])
    
    sam=SAM(dat)
    sam.preprocess_data()
    sam.run()
    
    sam.adata.obs_names_make_unique()
    sam.adata.var_names_make_unique()

    sam1.adata.obs_names_make_unique()
    sam1.adata.var_names_make_unique()
    
    sams = {'mg':sam,'ac':sam1}

    sm = SAMAP(
        sams,
        f_maps = '../../BLASTMAPPING/Hypo_proj/',
    )
    
    sm.run(pairwise=True)
    
    save_samap(sm , '../../sm_Allen_Full_ac_ncbi_soupx_cleaned_08272026_subclass_250_'+str(i)+'.pkl')

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8148776508874545
Computing the UMAP embedding...
Elapsed time: 543.1905598640442 seconds
Not updating the manifold...
Not updating the manifold...
19480 `mg` gene symbols match between the datasets and the BLAST graph.
18953 `ac` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 201.99112558364868
Correcting data with means. 376.72657680511475
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species ac...
Indegree coarsening
0/7 (0, 126101)
1/7 (20000, 126101)
2/7 (40000, 126101)
3/7 (60000, 126101)
4/7 (80000, 126101)
5/7 (100000, 126101)
6/7 (120000, 126101)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5337569488992522 
Max A.S. improvement: 0.982721025397059 
Min A.S. improvement: 0.0
Ca

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8146503635320411
Computing the UMAP embedding...
Elapsed time: 632.7697539329529 seconds
Not updating the manifold...
19480 `mg` gene symbols match between the datasets and the BLAST graph.
18953 `ac` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 400.45238637924194
Correcting data with means. 417.8025543689728
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species ac...
Indegree coarsening
0/7 (0, 126099)
1/7 (20000, 126099)
2/7 (40000, 126099)
3/7 (60000, 126099)
4/7 (80000, 126099)
5/7 (100000, 126099)
6/7 (120000, 126099)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.532555518220511 
Max A.S. improvement: 0.9692125625402349 
Min A.S. improvement: 0.0
Calculating gene-gene correlatio

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8149139489237648
Computing the UMAP embedding...
Elapsed time: 595.2069191932678 seconds
Not updating the manifold...
19480 `mg` gene symbols match between the datasets and the BLAST graph.
18953 `ac` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 281.1868715286255
Correcting data with means. 279.3162271976471
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species ac...
Indegree coarsening
0/7 (0, 126094)
1/7 (20000, 126094)
2/7 (40000, 126094)
3/7 (60000, 126094)
4/7 (80000, 126094)
5/7 (100000, 126094)
6/7 (120000, 126094)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5332768857578916 
Max A.S. improvement: 0.9771208891684763 
Min A.S. improvement: 0.0
Calculating gene-gene correlatio

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8149148967952387
Computing the UMAP embedding...
Elapsed time: 432.1300415992737 seconds
Not updating the manifold...
19480 `mg` gene symbols match between the datasets and the BLAST graph.
18953 `ac` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 350.26427364349365
Correcting data with means. 415.3875832557678
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species ac...
Indegree coarsening
0/7 (0, 126100)
1/7 (20000, 126100)
2/7 (40000, 126100)
3/7 (60000, 126100)
4/7 (80000, 126100)
5/7 (100000, 126100)
6/7 (120000, 126100)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5336773844009752 
Max A.S. improvement: 0.9829950787874624 
Min A.S. improvement: 0.0
Calculating gene-gene correlati

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


RUNNING SAM
Iteration: 0, Convergence: 1.0
Iteration: 1, Convergence: 0.8147478603757788
Computing the UMAP embedding...
Elapsed time: 653.9705882072449 seconds
Not updating the manifold...
19480 `mg` gene symbols match between the datasets and the BLAST graph.
18953 `ac` gene symbols match between the datasets and the BLAST graph.
Prepping datasets for translation.
Translating feature spaces pairwise.
Projecting data into joint latent space. 422.574378490448
Correcting data with means. 267.44520330429077
Expanding neighbourhoods of species mg...
Expanding neighbourhoods of species ac...
Indegree coarsening
0/7 (0, 126101)
1/7 (20000, 126101)
2/7 (40000, 126101)
3/7 (60000, 126101)
4/7 (80000, 126101)
5/7 (100000, 126101)
6/7 (120000, 126101)
Rescaling edge weights by expression correlations.
Concatenating SAM objects...
ITERATION: 0 
Average alignment score (A.S.):  0.5297212932828634 
Max A.S. improvement: 0.9855695718580788 
Min A.S. improvement: 0.0
Calculating gene-gene correlatio